# Fake Review Detector with Scavio API

Cross-reference Amazon product ratings with YouTube reviewer opinions to detect suspicious reviews. Uses the Scavio search API and LangChain to flag products where Amazon star ratings don't match real reviewer sentiment.

**What you will learn:**
- Pull Amazon product data (ratings, review counts) with ScavioAmazonProduct
- Search YouTube for honest reviews with ScavioYouTubeSearch
- Analyze YouTube reviewer metadata to assess credibility
- Build a trust score report comparing Amazon vs YouTube sentiment

**Prerequisites:**
- Free Scavio API key (50 free credits (one-time)): https://dashboard.scavio.dev
- OpenAI API key

**Tools used:** ScavioAmazonSearch, ScavioAmazonProduct, ScavioYouTubeSearch, ScavioYouTubeMetadata

In [1]:
# pip install langchain langchain-openai langchain-scavio python-dotenv

In [2]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_scavio import (
    ScavioAmazonSearch,
    ScavioAmazonProduct,
    ScavioYouTubeSearch,
    ScavioYouTubeMetadata,
)

load_dotenv(override=True)

True

In [3]:
SYSTEM_PROMPT = """You are FakeReviewDetector, a consumer protection agent that cross-references Amazon ratings with YouTube reviewer opinions.

Workflow:
1. Take the user's product name or category.
2. Call ScavioAmazonSearch to find the product on Amazon.
3. Call ScavioAmazonProduct on the top result to get full details:
   price, rating, review count, product features.
4. Call ScavioYouTubeSearch for "<product name> honest review" to find
   YouTube reviews of the same product.
5. Call ScavioYouTubeMetadata on the top 1-2 YouTube results to get
   view counts, like counts, and video descriptions.
6. Compare Amazon data vs YouTube reviewer signals and produce:

   ## Review Trust Report: <product name>

   ### Amazon Data
   - ASIN: <asin>
   - Price: $X.XX
   - Amazon Rating: X.X stars (X reviews)

   ### YouTube Reviewer Signals
   For each reviewed video:
   - Video: <title> by <channel>
   - Views: X | Likes: X
   - Description signals: <positive/negative points from description>

   ### Trust Analysis
   - Rating match: Do YouTube reviewers agree with Amazon's star rating?
   - Red flags: <any mismatches, suspicious patterns, or concerns>
   - Confidence: <High/Medium/Low>

   ### Verdict
   One paragraph: should the consumer trust these reviews?

Rules:
- Never invent ASINs, prices, ratings, or video data. Only use tool output.
- Call only ONE tool per step.
- Keep the final report under 400 words.
"""

In [4]:
def build_agent():
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    tools = [
        ScavioAmazonSearch(max_results=3),
        ScavioAmazonProduct(),
        ScavioYouTubeSearch(max_results=3),
        ScavioYouTubeMetadata(),
    ]
    return create_agent(model, tools=tools, system_prompt=SYSTEM_PROMPT)

In [5]:
agent = build_agent()
result = agent.invoke({
    "messages": [{"role": "user", "content": "Beats Solo 4 wireless headphones"}]
})
print(result["messages"][-1].content)

## Review Trust Report: Beats Solo 4 wireless headphones

### Amazon Data
- ASIN: B0CZPLV566
- Price: $149.95
- Amazon Rating: 4.6 stars (26,300 reviews)
- Key Features: Up to 50-hour battery life, ultralight ergonomic design, powerful and balanced sound, personalized spatial audio, fast charging, high-quality call performance, Class 1 Bluetooth.

### YouTube Reviewer Signals

1. Video: "AUDIO ENGINEER Reviews the BEATS SOLO 4 and Tests it Against the BEATS STUDIO PRO" by This is Tech Today
   - Views: 250,301 | Likes: 3,672
   - Description signals: Detailed technical review from an audio engineer perspective. Covers hardware, comfort, connection, battery, mic and call quality, sound quality with samples, and personal preferences. The review is unbiased and thorough, highlighting the Beats Solo 4 as a detour from older Beats models with a focus on balanced sound and spatial audio.

2. Video: "Two major problems: Beats Solo 4 [review]" by ORBIT
   - Views: 30,873 | Likes: 378
   - Desc

## Next Steps

- Analyze any Amazon product's review trustworthiness
- Compare review patterns across competing products
- Build a browser extension that flags suspicious products
- Combine with ScavioWalmartProduct for cross-platform review comparison

**Credits used:** ~5-7 per run (Amazon search + product + YouTube search + metadata)